In [1]:
from hullwhite import GeneratePathsHWAndAssetEuler, P0T, PriceEuropeanOptionHWMC
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

## Cálculo de Theta mediante diferencias finitas

Theta, mide cómo varía el precio de la opción con el paso del tiempo.

En la formulación clásica, la Theta se define como la derivada del precio respecto al tiempo actual. Sin embargo, al trabajar en el instante inicial, resulta más natural aproximarla a partir de variaciones en el tiempo restante hasta vencimiento $T$.

$$
\Theta = -\frac{\partial V}{\partial T}
$$

$$
\Theta \approx -\,\frac{V(T+h)-V(T-h)}{2h}
$$


La Theta suele ser negativa para opciones europeas vanilla, ya que la pérdida de tiempo hasta vencimiento reduce el valor temporal de la opcionalidad.

A diferencia de otras sensibilidades, la Theta resulta delicada desde el punto de vista numérico, ya que modificar $T$ afecta simultáneamente a:
- la duración de la simulación;
- el descuento acumulado;
- la distribución final del activo;
- y, en consecuencia, al valor total de la opción.

In [2]:
def ThetaFD_HW_MC(NoOfPaths, NoOfSteps, T, P0T, lambd, eta, S0, sigma_S, rho, K,
                  h, option_type, seed):
    """
    Calcula la Theta por diferencias finitas centradas
    en el modelo Hull–White + activo.
    """

    if T - h <= 0:
        raise ValueError("T - h debe ser positivo.")

    # Precio con T + h
    paths_up = GeneratePathsHWAndAssetEuler(
        NoOfPaths=NoOfPaths,
        NoOfSteps=NoOfSteps,
        T=T + h,
        P0T=P0T,
        lambd=lambd,
        eta=eta,
        S0=S0,
        sigma_S=sigma_S,
        rho=rho,
        seed=seed
    )

    price_up, _ = PriceEuropeanOptionHWMC(
        paths=paths_up,
        K=K,
        T=T + h,
        option_type=option_type
    )

    # Precio con T - h
    paths_down = GeneratePathsHWAndAssetEuler(
        NoOfPaths=NoOfPaths,
        NoOfSteps=NoOfSteps,
        T=T - h,
        P0T=P0T,
        lambd=lambd,
        eta=eta,
        S0=S0,
        sigma_S=sigma_S,
        rho=rho,
        seed=seed
    )

    price_down, _ = PriceEuropeanOptionHWMC(
        paths=paths_down,
        K=K,
        T=T - h,
        option_type=option_type
    )

    theta_fd = -(price_up - price_down) / (2.0 * h)

    return theta_fd, price_up, price_down

In [3]:
# Parámetros del modelo
NoOfPaths = 20000
NoOfSteps = 250
T = 1.0

lambd = 0.5
eta = 0.01

# Parámetros del activo
S0 = 100.0
sigma_S = 0.20
rho = -0.3

# Parámetros de la opción
K = 100.0
option_type = "call"

# Bump temporal: un día
h_T = 1 / 365

theta_fd, price_T_up, price_T_down = ThetaFD_HW_MC(
    NoOfPaths=NoOfPaths,
    NoOfSteps=NoOfSteps,
    T=T,
    P0T=P0T,
    lambd=lambd,
    eta=eta,
    S0=S0,
    sigma_S=sigma_S,
    rho=rho,
    K=K,
    h=h_T,
    option_type=option_type,
    seed=123
)

print(f"Precio con T + h = {price_T_up:.6f}")
print(f"Precio con T - h = {price_T_down:.6f}")
print(f"Theta FD         = {theta_fd:.6f}")

Precio con T + h = 10.401531
Precio con T - h = 10.366750
Theta FD         = -6.347589


#### Resultado numérico para Theta por diferencias finitas

El valor obtenido para la Theta es negativo. Esto refleja la pérdida de valor temporal de la opción a medida que transcurre el tiempo.

Dado que la Theta se ha calculado respecto al vencimiento residual expresado en años, su interpretación natural es anual. En este caso, una Theta aproximada de $-6.35$ indica que, en términos anuales, el precio de la opción disminuiría en unas 6.35 unidades al reducirse el tiempo restante hasta vencimiento.

Expresado en términos diarios, esto equivale aproximadamente a:

$$
\Theta_{\text{diaria}} \approx \frac{-6.347589}{365} \approx -0.0174
$$

lo que significa que la opción pierde alrededor de 1.7 céntimos por día, manteniendo constantes el resto de parámetros.